# Install required packages

In [ ]:
!pip install groq requests duckduckgo-search -q
!pip install \
    langchain==0.2.17 \
    langchain-core==0.2.43 \
    langchain-community==0.2.19 \
    langfuse==2.60.2
!pip install \
langgraph==0.2.60


# Import necessary libraries

In [ ]:
from google.colab import userdata
import uuid
import requests
from langfuse.callback import CallbackHandler
from langfuse import Langfuse
from langfuse.decorators import observe
from langgraph.graph import StateGraph, END
from groq import Groq
from duckduckgo_search import DDGS

# Retrieve API keys from Colab Secrets using userdata

In [ ]:
try:
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
    LANGFUSE_SECRET_KEY = userdata.get("LANGFUSE_API_KEY")
    LANGFUSE_PUBLIC_KEY = userdata.get("LANGFUSE_PUBLIC_KEY")
    LANGFUSE_HOST = userdata.get("LANGFUSE_HOST")
except Exception as e:
    print(f"Error retrieving API keys from Colab Secrets: {e}")
    raise ValueError("Ensure GROQ_API_KEY, LANGFUSE_API_KEY, LANGFUSE_PUBLIC_KEY, and LANGFUSE_HOST are set in Colab Secrets.")


# Debug: Print keys to verify retrieval

(remove in production for security)

In [ ]:
print("Debug: Verifying API keys retrieval...")
print(f"GROQ_API_KEY: {'Set' if GROQ_API_KEY else 'Not set'}")
print(f"LANGFUSE_PUBLIC_KEY: {'Set' if LANGFUSE_PUBLIC_KEY else 'Not set'}")
print(f"LANGFUSE_SECRET_KEY: {'Set' if LANGFUSE_SECRET_KEY else 'Not set'}")
print(f"LANGFUSE_HOST: {LANGFUSE_HOST if LANGFUSE_HOST else 'Not set'}")

Debug: Verifying API keys retrieval...
GROQ_API_KEY: Set
LANGFUSE_PUBLIC_KEY: Set
LANGFUSE_SECRET_KEY: Set
LANGFUSE_HOST: https://cloud.langfuse.com


# Validate that API keys are set

In [ ]:
if not all([GROQ_API_KEY, LANGFUSE_SECRET_KEY, LANGFUSE_PUBLIC_KEY, LANGFUSE_HOST]):
    raise ValueError("One or more API keys are missing in Colab Secrets.")

# List available models to confirm valid model IDs

In [ ]:
print("Listing available models from Groq API...")
try:
    url = "https://api.groq.com/openai/v1/models"
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }
    response = requests.get(url, headers=headers)
    models = response.json()
    print("Available models:", [model['id'] for model in models['data']])
except Exception as e:
    print(f"Error listing models: {e}")
    raise

Listing available models from Groq API...
Available models: ['meta-llama/llama-prompt-guard-2-22m', 'meta-llama/llama-prompt-guard-2-86m', 'groq/compound', 'groq/compound-mini', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-20b', 'llama-3.1-8b-instant', 'qwen/qwen3.6-27b', 'canopylabs/orpheus-arabic-saudi', 'whisper-large-v3', 'whisper-large-v3-turbo', 'openai/gpt-oss-safeguard-20b', 'allam-2-7b', 'openai/gpt-oss-120b', 'llama-3.3-70b-versatile']


# Test Groq API key with a simple request

In [ ]:
print("Testing Groq API key...")
try:
    groq_client = Groq(api_key=GROQ_API_KEY)
    test_response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "Hello"}],
        max_tokens=10
    )
    print("Groq API key test successful:", test_response.choices[0].message.content)
except Exception as e:
    print(f"Groq API key test failed: {e}")
    raise

Testing Groq API key...
Groq API key test successful: Hello. How can I assist you today?


# Initialize Langfuse callback handler

In [ ]:
try:
    langfuse_handler = CallbackHandler(
        secret_key=LANGFUSE_SECRET_KEY,
        public_key=LANGFUSE_PUBLIC_KEY,
        host=LANGFUSE_HOST
    )
except Exception as e:
    print(f"Error initializing Langfuse handler: {e}")
    raise

# Initialize Langfuse client for manual scoring

In [ ]:
try:
    langfuse = Langfuse(
        public_key=LANGFUSE_PUBLIC_KEY,
        secret_key=LANGFUSE_SECRET_KEY,
        host=LANGFUSE_HOST
    )
except Exception as e:
    print(f"Error initializing Langfuse client: {e}")
    raise

# Session management for tracing

In [ ]:
session_id = None
def set_new_session_id():
    global session_id
    session_id = str(uuid.uuid4())

# Initialize session
set_new_session_id()

# Define state for LangGraph

In [ ]:
from typing import TypedDict
class GraphState(TypedDict):
    user_input: str
    search_results: str
    analysis: str
    summary: str

# Node 1: Search Agent

Searches DuckDuckGo for information on the topic

In [ ]:
@observe()
def search_agent(state: GraphState) -> GraphState:
    from langfuse.decorators import langfuse_context
    user_input = state["user_input"]
    search_query = f"{user_input} recent developments"

    # Update observation metadata
    langfuse_context.update_current_observation(
        input=search_query,
        metadata={"organization": "agenticai", "project": "Langfuse_demo", "agent": "search_agent"}
    )

    # Perform DuckDuckGo search
    try:
        with DDGS() as ddgs:
            results = ddgs.text(keywords=search_query, max_results=3)
            search_results = "\n".join([result['body'] for result in results]) if results else "No search results found."
    except Exception as e:
        print(f"Error during DuckDuckGo search: {e}")
        search_results = "Failed to retrieve search results."

    return {"user_input": user_input, "search_results": search_results}

# Node 2: Analyzer Agent

Analyzes search results to identify key points

In [ ]:
@observe(as_type="generation")
def analyzer_agent(state: GraphState) -> GraphState:
    from langfuse.decorators import langfuse_context
    user_input = state["user_input"]
    search_results = state["search_results"]

    # Update observation metadata
    langfuse_context.update_current_observation(
        input=f"User input: {user_input}\nSearch results: {search_results}",
        model="llama-3.3-70b-versatile",
        metadata={"organization": "agenticai", "project": "Langfuse_demo", "agent": "analyzer_agent"}
    )

    # Call Groq API to analyze the search results
    prompt = f"Analyze the following search results for the topic '{user_input}' and identify key points or trends:\n{search_results}"
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an AI that analyzes information and identifies key points or trends."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=200
    )
    analysis = response.choices[0].message.content

    # Update usage details
    langfuse_context.update_current_observation(
        usage_details={
            "input": response.usage.prompt_tokens,
            "output": response.usage.completion_tokens
        }
    )

    return {"user_input": user_input, "search_results": search_results, "analysis": analysis}

# Node 3: Summarizer Agent

Summarizes the analysis into a concise report

In [ ]:
@observe(as_type="generation")
def summarizer_agent(state: GraphState) -> GraphState:
    from langfuse.decorators import langfuse_context
    user_input = state["user_input"]
    analysis = state["analysis"]

    # Update observation metadata
    langfuse_context.update_current_observation(
        input=f"User input: {user_input}\nAnalysis: {analysis}",
        model="llama-3.3-70b-versatile",
        metadata={"organization": "agenticai", "project": "Langfuse_demo", "agent": "summarizer_agent"}
    )

    # Call Groq API to summarize the analysis
    prompt = f"Summarize the following analysis for the topic '{user_input}' into a concise research report (100 tokens max):\n{analysis}"
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are an AI that summarizes information into concise reports."},
            {"role": "user", "content": prompt}
        ],
        max_tokens=100
    )
    summary = response.choices[0].message.content

    # Update usage details
    langfuse_context.update_current_observation(
        usage_details={
            "input": response.usage.prompt_tokens,
            "output": response.usage.completion_tokens
        }
    )

    return {
        "user_input": user_input,
        "search_results": state["search_results"],
        "analysis": analysis,
        "summary": summary
    }

# Define the LangGraph workflow

In [ ]:
workflow = StateGraph(GraphState)

# Add nodes for each agent
workflow.add_node("search_agent", search_agent)
workflow.add_node("analyzer_agent", analyzer_agent)
workflow.add_node("summarizer_agent", summarizer_agent)

# Define edges: sequential flow from search → analyzer → summarizer
workflow.set_entry_point("search_agent")
workflow.add_edge("search_agent", "analyzer_agent")
workflow.add_edge("analyzer_agent", "summarizer_agent")
workflow.add_edge("summarizer_agent", END)

# Compile the graph with Langfuse tracing

In [ ]:
app = workflow.compile()
app = app.with_config({"callbacks": [langfuse_handler]})

# Main function

In [ ]:
def main():
    # Step 1: Get dynamic user input
    user_input = input("Enter a research topic (e.g., AI advancements, quantum computing): ").strip()
    if not user_input:
        user_input = "AI advancements"  # Default if input is empty
        print("No input provided, defaulting to 'AI advancements'.")

    print(f"Researching topic: {user_input}")
    try:
        result = app.invoke(
            {"user_input": user_input},
            config={
                "metadata": {
                    "session_id": session_id,
                    "organization": "agenticai",
                    "project": "Langfuse_demo"
                }
            }
        )
        print("Search Results:", result["search_results"])
        print("Analysis:", result["analysis"])
        print("Research Summary:", result["summary"])
    except Exception as e:
        print(f"Error during graph invocation: {e}")
        result = {"summary": "Failed to generate research summary due to an error."}
        print("Research Summary:", result["summary"])

    # Step 2: Create a trace for evaluation
    trace = langfuse.trace(
        name="multi-agent-research-langgraph",
        user_id="demo-sam-001",
        session_id=session_id,
        metadata={
            "organization": "agenticai",
            "project": "Langfuse_demo",
            "input_tokens": len(user_input.split()),
            "output_tokens": len(result["summary"].split())
        }
    )

    # Step 3: Perform a basic evaluation by scoring the trace
    trace.score(
        name="response-quality",
        value=0.9,  # Example score (0 to 1)
        comment="The research summary was concise and informative."
    )

    # Ensure all events are sent to Langfuse
    langfuse.flush()

# Run the demo
if __name__ == "__main__":
    main()

# Instructions to view the trace in Langfuse Dashboard
print("Go to the Langfuse dashboard to view the trace:")
print("URL: https://cloud.langfuse.com")
print("Organization: agenticai")
print("Project: Langfuse_demo")
print("Look under the 'Traces' tab for 'multi-agent-research-langgraph' and check 'Sessions' for session tracking.")

Streaming output truncated to the last 5000 lines.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replac

Enter a research topic (e.g., AI advancements, quantum computing): AI advancements


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag